# Benchmark Gemini — Devis IA (autonome)

Aucun script Python externe n’est requis. Ajoutez comme Kaggle Inputs uniquement les deux fichiers JSON de données et créez le secret `GEMINI_API_KEY`.


In [ ]:
!pip -q install google-genai scipy


In [ ]:
# ÉVALUATEUR MÉTIER GEPA — copie directe et inchangée depuis optimisationGepa.py.
from __future__ import annotations
import json
import math
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable
import numpy as np
from scipy.optimize import linear_sum_assignment

INITIAL_INSTRUCTIONS = """Tu transformes une description de travaux BTP en prestations de devis.
Utilise exclusivement les IDs du catalogue fourni dans le contexte.
S�lectionne une prestation uniquement si elle est explicitement demand�e ou d�ductible sans ambigu�t�.
Pour chaque quantit� non pr�sente dans la description, retourne null. Ne d�duis une quantit� que si le calcul est directement justifi� par les dimensions donn�es.
N'invente ni prestation, ni option, ni choix ; ne s�lectionne jamais une option par d�faut.
R�ponds uniquement avec un JSON strictement conforme � ce sch�ma :
{"prestations":[{"prestation_id":1,"quantite_ouvrage":8,"options":[{"option_id":1,"choix_id":2}]}]}
La racine contient uniquement "prestations". Une prestation contient uniquement "prestation_id", "quantite_ouvrage" et "options". Une option contient uniquement "option_id" et "choix_id"."""


def load_json(path: str | Path) -> Any:
    with Path(path).open(encoding="utf-8") as stream:
        return json.load(stream)


def f1(counts: dict[str, int]) -> float:
    denominator = 2 * counts["tp"] + counts["fp"] + counts["fn"]
    return 1.0 if denominator == 0 else (2 * counts["tp"]) / denominator


def empty_counts() -> dict[str, int]:
    return {"tp": 0, "fp": 0, "fn": 0}


def new_metric_counts() -> dict[str, dict[str, int]]:
    return {
        "prestation_id_f1": empty_counts(),
        "quantite_ouvrage_f1": empty_counts(),
        "options_choix_f1": empty_counts(),
    }


def values_equal(expected: Any, predicted: Any) -> bool:
    """`null == null` est un vrai positif ; bool n'est jamais une quantit�."""
    return expected == predicted


@dataclass
class CatalogueIndex:
    prestations: dict[int, dict[str, Any]]
    options: dict[int, dict[str, Any]]
    choix: dict[int, dict[str, Any]]
    prompt_catalogue: list[dict[str, Any]]

    @classmethod
    def from_export(cls, export: dict[str, Any]) -> "CatalogueIndex":
        prestations = {item["id"]: item for item in export["prestations"]}
        options = {item["id"]: item for item in export["options"]}
        choix = {item["id"]: item for item in export["choix"]}

        by_prestation: dict[int, list[dict[str, Any]]] = {key: [] for key in prestations}
        by_option: dict[int, list[dict[str, Any]]] = {key: [] for key in options}
        for option in options.values():
            if option["prestation_id"] not in prestations:
                raise ValueError(f"Option orpheline dans le catalogue : {option['id']}")
            by_prestation[option["prestation_id"]].append(option)
        for choice in choix.values():
            if choice["option_id"] not in options:
                raise ValueError(f"Choix orphelin dans le catalogue : {choice['id']}")
            by_option[choice["option_id"]].append(choice)

        prompt_catalogue = []
        for prestation in prestations.values():
            prompt_catalogue.append(
                {
                    "id": prestation["id"],
                    "nom": prestation["nom"],
                    "description": prestation.get("description", ""),
                    "options": [
                        {
                            "id": option["id"],
                            "nom": option["nom"],
                            "description": option.get("description", ""),
                            "choix": [
                                {
                                    "id": choice["id"],
                                    "nom": choice["nom"],
                                    "description": choice.get("description", ""),
                                }
                                for choice in by_option[option["id"]]
                            ],
                        }
                        for option in by_prestation[prestation["id"]]
                    ],
                }
            )
        return cls(prestations, options, choix, prompt_catalogue)

    def reference_errors(self, prediction: dict[str, Any]) -> list[str]:
        errors: list[str] = []
        for position, prestation in enumerate(prediction.get("prestations", [])):
            prestation_id = prestation["prestation_id"]
            if prestation_id not in self.prestations:
                errors.append(f"prestations[{position}].prestation_id={prestation_id} absent du catalogue")
            for option_position, option in enumerate(prestation["options"]):
                option_id = option["option_id"]
                choice_id = option["choix_id"]
                if option_id not in self.options:
                    errors.append(f"prestations[{position}].options[{option_position}].option_id={option_id} absent du catalogue")
                    continue
                if self.options[option_id]["prestation_id"] != prestation_id:
                    errors.append(
                        f"option {option_id} n'appartient pas � la prestation {prestation_id}"
                    )
                if choice_id not in self.choix:
                    errors.append(f"choix_id={choice_id} absent du catalogue")
                elif self.choix[choice_id]["option_id"] != option_id:
                    errors.append(f"choix {choice_id} n'appartient pas � l'option {option_id}")
        return errors


def validate_output(raw_text: str) -> tuple[dict[str, Any] | None, list[str]]:
    """Valide la structure JSON sans m�langer conformit� et exactitude m�tier."""
    try:
        value = json.loads(raw_text)
    except Exception as error:
        return None, [f"JSON invalide : {error}"]

    errors: list[str] = []
    if not isinstance(value, dict) or set(value) != {"prestations"}:
        return None, ["La racine doit �tre un objet avec uniquement la cl� 'prestations'."]
    if not isinstance(value["prestations"], list):
        return None, ["'prestations' doit �tre un tableau."]

    for index, prestation in enumerate(value["prestations"]):
        prefix = f"prestations[{index}]"
        if not isinstance(prestation, dict) or set(prestation) != {
            "prestation_id", "quantite_ouvrage", "options"
        }:
            errors.append(f"{prefix} doit contenir uniquement prestation_id, quantite_ouvrage, options.")
            continue
        if type(prestation["prestation_id"]) is not int:
            errors.append(f"{prefix}.prestation_id doit �tre un entier.")
        quantity = prestation["quantite_ouvrage"]
        if quantity is not None and (type(quantity) not in (int, float) or not math.isfinite(quantity)):
            errors.append(f"{prefix}.quantite_ouvrage doit �tre un nombre fini ou null.")
        if not isinstance(prestation["options"], list):
            errors.append(f"{prefix}.options doit �tre un tableau.")
            continue
        for option_index, option in enumerate(prestation["options"]):
            option_prefix = f"{prefix}.options[{option_index}]"
            if not isinstance(option, dict) or set(option) != {"option_id", "choix_id"}:
                errors.append(f"{option_prefix} doit contenir uniquement option_id et choix_id.")
                continue
            if type(option["option_id"]) is not int or type(option["choix_id"]) is not int:
                errors.append(f"{option_prefix}.option_id et choix_id doivent �tre des entiers.")
    return (value if not errors else None), errors


def _assignment(
    expected: list[dict[str, Any]],
    predicted: list[dict[str, Any]],
    paired_cost: Callable[[dict[str, Any], dict[str, Any]], int],
    expected_missing_cost: Callable[[dict[str, Any]], int],
    predicted_extra_cost: Callable[[dict[str, Any]], int],
) -> list[tuple[int | None, int | None]]:
    """Assignment global avec colonnes/lignes factices pour les �l�ments absents/en trop."""
    expected_count, predicted_count = len(expected), len(predicted)
    if expected_count == predicted_count == 0:
        return []
    size = expected_count + predicted_count
    costs = np.zeros((size, size), dtype=float)
    if expected_count and predicted_count:
        costs[:expected_count, :predicted_count] = [
            [paired_cost(item, other) for other in predicted] for item in expected
        ]
    if expected_count:
        costs[:expected_count, predicted_count:] = [
            [expected_missing_cost(item)] * expected_count for item in expected
        ]
    if predicted_count:
        costs[expected_count:, :predicted_count] = [
            [predicted_extra_cost(item)] * predicted_count for item in predicted
        ]

    rows, columns = linear_sum_assignment(costs)
    return [
        (
            int(row) if row < expected_count else None,
            int(column) if column < predicted_count else None,
        )
        for row, column in zip(rows, columns)
        if row < expected_count or column < predicted_count
    ]


def _append_cell(
    counts: dict[str, dict[str, int]],
    metric: str,
    expected: Any,
    predicted: Any,
    path: str,
    missing: list[dict[str, Any]],
    incorrect: list[dict[str, Any]],
) -> None:
    if values_equal(expected, predicted):
        counts[metric]["tp"] += 1
    else:
        counts[metric]["fn"] += 1
        counts[metric]["fp"] += 1
        incorrect.append({"path": path, "expected": expected, "predicted": predicted})


def _missing_prestation(
    item: dict[str, Any],
    counts: dict[str, dict[str, int]],
    missing: list[dict[str, Any]],
    label: str,
) -> None:
    counts["prestation_id_f1"]["fn"] += 1
    counts["quantite_ouvrage_f1"]["fn"] += 1
    counts["options_choix_f1"]["fn"] += 2 * len(item["options"])
    missing.append({"path": label, "expected": item})


def _extra_prestation(
    item: dict[str, Any],
    counts: dict[str, dict[str, int]],
    extra: list[dict[str, Any]],
    label: str,
) -> None:
    counts["prestation_id_f1"]["fp"] += 1
    counts["quantite_ouvrage_f1"]["fp"] += 1
    counts["options_choix_f1"]["fp"] += 2 * len(item["options"])
    extra.append({"path": label, "predicted": item})


def _score_options(
    expected: list[dict[str, Any]],
    predicted: list[dict[str, Any]],
    counts: dict[str, dict[str, int]],
    missing: list[dict[str, Any]],
    incorrect: list[dict[str, Any]],
    extra: list[dict[str, Any]],
    prefix: str,
) -> None:
    def option_pair_cost(left: dict[str, Any], right: dict[str, Any]) -> int:
        return int(left["option_id"] != right["option_id"]) + int(left["choix_id"] != right["choix_id"])

    pairs = _assignment(expected, predicted, option_pair_cost, lambda _: 2, lambda _: 2)
    for expected_index, predicted_index in pairs:
        if expected_index is None:
            counts["options_choix_f1"]["fp"] += 2
            extra.append({"path": f"{prefix}.options[{predicted_index}]", "predicted": predicted[predicted_index]})
        elif predicted_index is None:
            counts["options_choix_f1"]["fn"] += 2
            missing.append({"path": f"{prefix}.options[{expected_index}]", "expected": expected[expected_index]})
        else:
            expected_item, predicted_item = expected[expected_index], predicted[predicted_index]
            _append_cell(
                counts, "options_choix_f1", expected_item["option_id"], predicted_item["option_id"],
                f"{prefix}.options[{expected_index}].option_id", missing, incorrect,
            )
            _append_cell(
                counts, "options_choix_f1", expected_item["choix_id"], predicted_item["choix_id"],
                f"{prefix}.options[{expected_index}].choix_id", missing, incorrect,
            )


def option_alignment_cost(expected: list[dict[str, Any]], predicted: list[dict[str, Any]]) -> int:
    """Nombre minimal de cellules option/choix discordantes, apr�s Hongrois."""
    def pair_cost(left: dict[str, Any], right: dict[str, Any]) -> int:
        return int(left["option_id"] != right["option_id"]) + int(left["choix_id"] != right["choix_id"])

    total = 0
    for expected_index, predicted_index in _assignment(expected, predicted, pair_cost, lambda _: 2, lambda _: 2):
        if expected_index is None or predicted_index is None:
            total += 2
        else:
            total += pair_cost(expected[expected_index], predicted[predicted_index])
    return total


def score_prediction(
    expected: dict[str, Any],
    predicted: dict[str, Any] | None,
    json_errors: list[str],
    catalogue: CatalogueIndex,
) -> dict[str, Any]:
    """�value une pr�diction par alignement hongrois sur les prestations et options."""
    counts = new_metric_counts()
    missing: list[dict[str, Any]] = []
    incorrect: list[dict[str, Any]] = []
    extra: list[dict[str, Any]] = []
    expected_items = expected["prestations"]
    predicted_items = [] if predicted is None else predicted["prestations"]

    def prestation_pair_cost(left: dict[str, Any], right: dict[str, Any]) -> int:
        return (
            int(left["prestation_id"] != right["prestation_id"])
            + int(left["quantite_ouvrage"] != right["quantite_ouvrage"])
            + option_alignment_cost(left["options"], right["options"])
        )

    pairs = _assignment(
        expected_items,
        predicted_items,
        prestation_pair_cost,
        lambda item: 2 + 2 * len(item["options"]),
        lambda item: 2 + 2 * len(item["options"]),
    )
    for expected_index, predicted_index in pairs:
        if expected_index is None:
            _extra_prestation(predicted_items[predicted_index], counts, extra, f"prestations[{predicted_index}]")
            continue
        if predicted_index is None:
            _missing_prestation(expected_items[expected_index], counts, missing, f"prestations[{expected_index}]")
            continue

        expected_item, predicted_item = expected_items[expected_index], predicted_items[predicted_index]
        prefix = f"prestations[{expected_index}]"
        _append_cell(
            counts, "prestation_id_f1", expected_item["prestation_id"], predicted_item["prestation_id"],
            f"{prefix}.prestation_id", missing, incorrect,
        )
        _append_cell(
            counts, "quantite_ouvrage_f1", expected_item["quantite_ouvrage"], predicted_item["quantite_ouvrage"],
            f"{prefix}.quantite_ouvrage", missing, incorrect,
        )
        _score_options(
            expected_item["options"], predicted_item["options"], counts, missing, incorrect, extra, prefix
        )

    metrics = {name: {**value, "f1": f1(value)} for name, value in counts.items()}
    catalogue_errors = [] if predicted is None else catalogue.reference_errors(predicted)
    return {
        "json_conformity": 1.0 if predicted is not None else 0.0,
        "catalogue_references_valid": not catalogue_errors,
        "json_errors": json_errors,
        "catalogue_errors": catalogue_errors,
        "metrics": metrics,
        "missing": missing,
        "incorrect": incorrect,
        "extra": extra,
    }




In [ ]:
# BENCHMARK — fonctions nouvelles, séparées de l'évaluateur GEPA ci-dessus.
import csv, os, statistics, time
from pathlib import Path
from google import genai
from google.genai import types

def gemini_key():
    if os.getenv("GEMINI_API_KEY"): return os.getenv("GEMINI_API_KEY")
    from kaggle_secrets import UserSecretsClient
    return UserSecretsClient().get_secret("GEMINI_API_KEY")

def quotas(model):
    for key in ("rpm","tpm","rpd"):
        if type(model.get(key)) not in (int,float) or model[key]<=0: raise ValueError(f"Quota réel obligatoire : {model['name']}.{key}")
    return float(model["rpm"]),float(model["tpm"]),float(model["rpd"])

def call(client,model,prompt,description,interval):
    starts=getattr(call,"starts",{}); name=model["model_id"]
    if name in starts:
        wait=interval-(time.monotonic()-starts[name])
        if wait>0: time.sleep(wait)
    start=time.perf_counter(); starts[name]=time.monotonic(); call.starts=starts
    try:
        r=client.models.generate_content(model=name,contents=description,config=types.GenerateContentConfig(system_instruction=prompt,response_mime_type="application/json",temperature=0))
        u=getattr(r,"usage_metadata",None); tokens=getattr(u,"prompt_token_count",None)
        return {"api_success":True,"raw_model_output":r.text or "","input_tokens":tokens if type(tokens)is int and tokens>=0 else None,"latency_seconds":time.perf_counter()-start,"error":None}
    except Exception as error:
        return {"api_success":False,"raw_model_output":None,"input_tokens":None,"latency_seconds":time.perf_counter()-start,"error":f"{type(error).__name__}: {error}"}

def metier(report):
    m=report["metrics"]; return 100*(.8*m["prestation_id_f1"]["f1"]+.1*m["quantite_ouvrage_f1"]["f1"]+.1*m["options_choix_f1"]["f1"])

def benchmark(models,examples,catalogue,prompt,output):
    client=genai.Client(api_key=gemini_key(),http_options=types.HttpOptions(timeout=45_000)); summaries=[]
    for model in models:
        rpm,tpm,rpd=quotas(model); interval=max(float(model.get("minimum_interval_seconds",0)),60/rpm); runs=[]; details=[]
        print(f"\n[MODELE] {model['name']} — {model['model_id']} | RPM={rpm:g}, TPM={tpm:g}, RPD={rpd:g}")
        for pos,example in enumerate(examples,1):
            print(f"  [{pos}/{len(examples)}] {example['synthetic_id']}",flush=True); group=[]
            for attempt in range(1,11):
                x={"synthetic_id":example["synthetic_id"],"attempt":attempt,**call(client,model,prompt,example["description"],interval)}
                x["latency_conformity"]=x["api_success"] and x["latency_seconds"]<=30
                # Une erreur API/timeout n'obtient jamais un F1 métier.
                if x["api_success"]:
                    predicted,errors=validate_output(x["raw_model_output"]); report=score_prediction(example["ground_truth"],predicted,errors,catalogue)
                    x.update({"json_conformity":report["json_conformity"],"catalogue_references_valid":report["catalogue_references_valid"],"f1_prestation_id":report["metrics"]["prestation_id_f1"]["f1"],"f1_quantite_ouvrage":report["metrics"]["quantite_ouvrage_f1"]["f1"],"f1_options_choix":report["metrics"]["options_choix_f1"]["f1"],"score_metier":metier(report),"json_errors":report["json_errors"],"catalogue_errors":report["catalogue_errors"],"missing":report["missing"],"incorrect":report["incorrect"],"extra":report["extra"]})
                else: x.update({"json_conformity":0.0,"catalogue_references_valid":None,"f1_prestation_id":None,"f1_quantite_ouvrage":None,"f1_options_choix":None,"score_metier":None,"json_errors":[],"catalogue_errors":[],"missing":[],"incorrect":[],"extra":[]})
                group.append(x); runs.append(x); status="OK" if x["latency_conformity"] else ("ERREUR_API" if not x["api_success"] else "TARDIF")
                print(f"    essai {attempt}/10 : {status} | {x['latency_seconds']:.2f}s | métier={x['score_metier']}",flush=True)
            scores=[x["score_metier"] for x in group if x["score_metier"] is not None]
            details.append({"synthetic_id":example["synthetic_id"],"score_metier":statistics.fmean(scores) if scores else None,"score_json":100*statistics.fmean(x["json_conformity"] for x in group),"score_latence":100*statistics.fmean(x["latency_conformity"] for x in group),"executions_api_echouees":sum(not x["api_success"] for x in group)})
        scores=[x["score_metier"] for x in runs if x["score_metier"] is not None]; tokens=[x["input_tokens"] for x in runs if x["input_tokens"] is not None]
        summaries.append({"model":{"name":model["name"],"model_id":model["model_id"]},"quotas":{"rpm":rpm,"tpm":tpm,"rpd":rpd},"tokens_entree_moyens":statistics.fmean(tokens) if tokens else None,"score_metier":statistics.fmean(scores) if scores else None,"score_json":100*statistics.fmean(x["json_conformity"] for x in runs),"score_latence":100*statistics.fmean(x["latency_conformity"] for x in runs),"runs":runs,"per_description":details})
    capacities=[]
    for x in summaries:
        tokens=x["tokens_entree_moyens"]
        if tokens is None or tokens<=0: raise RuntimeError("Tokens d'entrée Gemini absents : capacité impossible.")
        q=x["quotas"]; c=min(q["rpd"],1440*min(q["rpm"],q["tpm"]/tokens)); capacities.append(c); x["capacite_theorique_devis_par_jour"]=c
    low,high=min(capacities),max(capacities)
    for x in summaries:
        c=x["capacite_theorique_devis_par_jour"]; x["score_capacite"]=100 if low==high else 100*(c-low)/(high-low)
        x["score_global"]=None if x["score_metier"] is None else .4*x["score_metier"]+.2*x["score_json"]+.2*x["score_latence"]+.2*x["score_capacite"]
    output=Path(output); output.mkdir(parents=True,exist_ok=True)
    (output/"benchmark_detailed_results.json").write_text(json.dumps({"models":summaries},ensure_ascii=False,indent=2),encoding="utf-8")
    compact=[{k:v for k,v in x.items() if k not in ("runs","per_description")} for x in summaries]
    (output/"benchmark_summary.json").write_text(json.dumps({"models":compact},ensure_ascii=False,indent=2),encoding="utf-8")
    fields=["model_name","model_id","synthetic_id","attempt","api_success","latency_seconds","latency_conformity","input_tokens","json_conformity","catalogue_references_valid","f1_prestation_id","f1_quantite_ouvrage","f1_options_choix","score_metier","error"]
    with (output/"benchmark_runs.csv").open("w",encoding="utf-8",newline="") as f:
        w=csv.DictWriter(f,fieldnames=fields); w.writeheader()
        for x in summaries:
            for row in x["runs"]: w.writerow({"model_name":x["model"]["name"],"model_id":x["model"]["model_id"],**{key:row.get(key) for key in fields[2:]}})
    return summaries


In [ ]:
import json
from pathlib import Path
def file(name):
    found=list(Path("/kaggle/input").rglob(name))
    if len(found)!=1: raise FileNotFoundError(f"Fichier attendu une seule fois : {name}. Trouvé : {found}")
    return str(found[0])
CATALOGUE=CatalogueIndex.from_export(load_json(file("catalogue_postgresql_ids_reels.json")))
TEST_EXAMPLES=load_json(file("test_dataset_descriptions_ground_truth.json"))["examples"]
for ex in TEST_EXAMPLES:
    if errors:=CATALOGUE.reference_errors(ex["ground_truth"]): raise ValueError(f"GT invalide pour {ex['synthetic_id']} : {errors}")
# Garde le prompt initial GEPA, ou remplace la valeur par ton prompt GEPA final.
INSTRUCTIONS=INITIAL_INSTRUCTIONS
# Renseignez les IDs et les quotas RÉELS. Aucune valeur n'est inventée.
MODELS=[
 {"name":"Gemini modèle 1","model_id":"REMPLACER_PAR_MODELE_1","rpm":None,"tpm":None,"rpd":None,"minimum_interval_seconds":0},
 {"name":"Gemini modèle 2","model_id":"REMPLACER_PAR_MODELE_2","rpm":None,"tpm":None,"rpd":None,"minimum_interval_seconds":0},
]
SYSTEM_PROMPT=f"{INSTRUCTIONS}\n\nCatalogue PostgreSQL disponible pour cette entreprise (IDs réels) :\n{json.dumps(CATALOGUE.prompt_catalogue,ensure_ascii=False)}"
OUTPUT_DIR="/kaggle/working/devis_ia_gemini_benchmark"
print(f"{len(TEST_EXAMPLES)} descriptions — {len(TEST_EXAMPLES)*10} appels par modèle.")


In [ ]:
# Secret Kaggle requis : GEMINI_API_KEY
results=benchmark(MODELS,TEST_EXAMPLES,CATALOGUE,SYSTEM_PROMPT,OUTPUT_DIR)
for x in results:
    print(json.dumps({key:x[key] for key in ("model","score_metier","score_json","score_latence","capacite_theorique_devis_par_jour","score_capacite","score_global")},ensure_ascii=False,indent=2))
print(f"Résultats dans : {OUTPUT_DIR}")